In [ ]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

from xgboost import XGBClassifier

from feature_extraction.features import extract_all_features



In [ ]:
TRAIN_DIR = "data/processed/train"
VAL_DIR = "data/processed/validation"
TEST_DIR = "data/processed/test"

CLASSES = [
    "machinery_fault",
    "glass_breaking",
    "alarm_siren",
    "vehicle_horn",
    "animal_sound",
    "gunshot",
    "panic_scream",
    "aggression",
    "person_asking_help",
    "background_noise",
]




In [ ]:
def build_dataset(split_dir):
    rows = []
    labels = []

    for cls in CLASSES:
        cls_dir = os.path.join(split_dir, cls)

        if not os.path.isdir(cls_dir):
            print("missing folder:", cls_dir)
            continue

        files = [f for f in os.listdir(cls_dir) if f.lower().endswith(".wav")]

        for fname in files:
            fpath = os.path.join(cls_dir, fname)
            try:
                feats = extract_all_features(fpath)
                rows.append(feats)
                labels.append(cls)
            except Exception as e:
                print("failed:", fpath, e)

        print(cls, "->", len(files), "files done")

    df = pd.DataFrame(rows)
    df["label"] = labels
    return df




In [ ]:
def prepare_features(train_df, val_df, test_df):
    le = LabelEncoder()
    le.fit(CLASSES)

    X_train = train_df.drop(columns=["label"]).values
    y_train = le.transform(train_df["label"].values)

    X_val = val_df.drop(columns=["label"]).values
    y_val = le.transform(val_df["label"].values)

    X_test = test_df.drop(columns=["label"]).values
    y_test = le.transform(test_df["label"].values)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    return (
        X_train_scaled,
        y_train,
        X_val_scaled,
        y_val,
        X_test_scaled,
        y_test,
        scaler,
        le,
    )




In [ ]:
def train_xgb(X_train, y_train):
    param_grid = {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }

    base_xgb = XGBClassifier(
        objective="multi:softprob",
        num_class=len(CLASSES),
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )

    grid = GridSearchCV(
        base_xgb,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=2,
    )

    grid.fit(X_train, y_train)

    print("best params:", grid.best_params_)
    print("best cv f1_macro:", grid.best_score_)

    return grid.best_estimator_




In [ ]:
def evaluate(model, X, y, le, label):
    pred = model.predict(X)

    print(f"\n{label} accuracy:", accuracy_score(y, pred))
    print(f"{label} macro f1:", f1_score(y, pred, average="macro"))
    print(classification_report(y, pred, target_names=le.classes_))

    return pred




In [ ]:
def main():
    train_df = build_dataset(TRAIN_DIR)
    val_df = build_dataset(VAL_DIR)
    test_df = build_dataset(TEST_DIR)

    os.makedirs("data/metadata", exist_ok=True)
    train_df.to_csv("data/metadata/train_features.csv", index=False)
    val_df.to_csv("data/metadata/val_features.csv", index=False)
    test_df.to_csv("data/metadata/test_features.csv", index=False)

    (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        scaler,
        le,
    ) = prepare_features(train_df, val_df, test_df)

    xgb_model = train_xgb(X_train, y_train)

    evaluate(xgb_model, X_val, y_val, le, "validation")
    test_pred = evaluate(xgb_model, X_test, y_test, le, "test")

    cm = confusion_matrix(y_test, test_pred)
    cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
    print("\nconfusion matrix:")
    print(cm_df)

    feat_names = train_df.drop(columns=["label"]).columns
    importances = pd.Series(xgb_model.feature_importances_, index=feat_names)
    print("\ntop 20 important features:")
    print(importances.sort_values(ascending=False).head(20))

    os.makedirs("python_models", exist_ok=True)
    joblib.dump(xgb_model, "python_models/xgb_model.pkl")
    joblib.dump(scaler, "python_models/xgb_scaler.pkl")
    joblib.dump(le, "python_models/label_encoder.pkl")

    print("\nsaved xgboost model, scaler and label encoder")




In [ ]:
if __name__ == "__main__":
    main()